In [ ]:
from transformers import pipeline
import torch

# Summarizer


In [ ]:
summarizer = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
import json

with open('../data/top_headlines_us_context.json', 'r') as f:
    headlines_with_context = json.load(f)


In [14]:
from transformers import logging

logging.set_verbosity_error()

In [ ]:
from datasets import Dataset
from tqdm import tqdm
import re

PROMPT_TEMPLATE = (
    "Summarize the news in 2-3 concise sentences.\n"
    "Title: {title}\n"
    "Description: {description}\n"
    "Context: {context}\n"
    "Content: {content}\n"
    "Summary:"
)

def _clip(text, max_chars=900):
    text = str(text or "")
    return text[:max_chars]

def extract_summary(text):
    match = re.search(r"Summary:(.*)", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

texts = [
    PROMPT_TEMPLATE.format(
        context=_clip(article.get('context', ''), 900),
        title=_clip(article.get('title', ''), 220),
        description=_clip(article.get('description', ''), 350),
        content=_clip(article.get('content', ''), 500),
    )
    for article in headlines_with_context
]
sources = [article.get('source', {}) for article in headlines_with_context]
urls = [article.get('url', '') for article in headlines_with_context]

dataset = Dataset.from_dict({"text": texts, "source": sources, "url": urls})

batch_size = 1
summaries = []
for i in tqdm(range(0, len(dataset), batch_size), desc="Batch summarizing"):
    batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
    batch_texts = list(batch['text'])
    batch_outputs = summarizer(
        batch_texts,
        max_new_tokens=72,
        do_sample=False,
        return_full_text=False,
    )
    for output, source_obj in zip(batch_outputs, batch['source']):
        if isinstance(output, list):
            summary_text = output[0]['generated_text'] if output and 'generated_text' in output[0] else str(output)
        else:
            summary_text = output.get('generated_text', str(output))
        clean_summary = extract_summary(summary_text)
        summaries.append({
            "Summary": clean_summary,
            "source": source_obj
        })

Batch summarizing: 100%|██████████| 3/3 [01:13<00:00, 24.60s/it]


In [18]:
for s in summaries:
    print(f"Summary:\n{s['Summary']}\nsource: {s['source']}\n")

Summary:
'Real Housewives of Atlanta' alum Kim Zolciak agreed to be questioned under oath by lawyers representing her boyfriend Kyle Mowitz's estranged wife, Jillian Green, in a divorce case involving over $100 million in assets. Kim will provide answers to questions about money, gifts, and loans given to her by Kyle during their relationship. The agreement includes keeping the transcript private and not sharing it with third parties.
source: {'id': None, 'name': 'TMZ'}

Summary:
President Donald Trump's administration is using the 1884 Supreme Court case Elk v. Wilkins to defend his plan to limit birthright citizenship. The ruling stated that Native Americans born within U.S. territory were not citizens and had the same status as children of foreign governments. However, this argument is strongly contested by the ACLU, who are leading the challenge to Trump's executive order. The Trump administration argues that the case gives the Supreme Court the opportunity to restore the original 

# Named Entity Recognition

In [25]:
ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=-1
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [26]:
def _clean_entity_text(text):
    text = text.replace(" ##", "").replace("##", "")
    return " ".join(text.split()).strip()

summary_entities = []
for s in summaries:
    summary_text = s['Summary']
    ner_results = ner(summary_text)

    entities = {}
    for ent in ner_results:
        ent_type = ent.get('entity_group', ent.get('entity', 'UNK'))
        ent_text = _clean_entity_text(ent.get('word', ''))
        if not ent_text:
            continue
        entities.setdefault(ent_type, set()).add(ent_text)

    entities = {k: sorted(list(v)) for k, v in entities.items()}
    summary_entities.append({
        'summary': summary_text,
        'entities': entities,
        'source': s['source']
    })

print(summary_entities[0])

{'summary': "'Real Housewives of Atlanta' alum Kim Zolciak agreed to be questioned under oath by lawyers representing her boyfriend Kyle Mowitz's estranged wife, Jillian Green, in a divorce case involving over $100 million in assets. Kim will provide answers to questions about money, gifts, and loans given to her by Kyle during their relationship. The agreement includes keeping the transcript private and not sharing it with third parties.", 'entities': {'ORG': ['Real Housewives of'], 'MISC': ['Atlanta'], 'PER': ['Jill', 'Kim', 'Kim Zolciak', 'Kyle', 'Kyle Mowitz', 'ian Green']}, 'source': {'id': None, 'name': 'TMZ'}}


# Parse Relationships

In [ ]:
# Reuse the existing `summarizer` model to extract relationships for KG use
import json
import re
import torch
from tqdm import tqdm
from datasets import Dataset

REL_PROMPT_TEMPLATE = """You are a strict JSON generator for relationship extraction.

TASK:
Extract factual relationships from the given summary.

OUTPUT CONTRACT (MANDATORY):
Return exactly one JSON object with this shape and no extra keys/text:
{{"relationships":[{{"subject":"...","predicate":"...","object":"...","confidence":0.0}}]}}

RULES:
- Return ONLY JSON. No markdown fences. No explanations.
- If no valid relationship exists, return {{"relationships":[]}}.
- Max 5 relationships.
- Use short canonical entity names.
- Predicate should be short snake_case (e.g., sued, acquired, located_in, won).
- confidence is a float between 0 and 1.
- Only include facts supported by the summary.

Summary:
{summary}

Entities (hints):
{entities}
"""

def _generated_text_from_output(output):
    if isinstance(output, list):
        if output and isinstance(output[0], dict) and "generated_text" in output[0]:
            return output[0]["generated_text"]
        return str(output)
    if isinstance(output, dict):
        return output.get("generated_text", str(output))
    return str(output)

def _normalize_rel(subject, predicate, obj, confidence=0.7):
    subject = str(subject).strip()
    predicate = str(predicate).strip().lower().replace(" ", "_")
    obj = str(obj).strip()
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.7
    confidence = max(0.0, min(1.0, confidence))
    if not (subject and predicate and obj):
        return None
    return {
        "subject": subject,
        "predicate": predicate,
        "object": obj,
        "confidence": confidence,
    }

def _extract_json_object(text):
    text = text.strip()

    fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if fenced:
        return fenced.group(1).strip()

    obj = re.search(r"\{.*\}", text, re.DOTALL)
    if obj:
        return obj.group(0).strip()

    return ""

def _parse_relationships(raw_text):
    json_block = _extract_json_object(raw_text)
    if not json_block:
        return []

    try:
        parsed = json.loads(json_block)
    except Exception:
        return []

    if not isinstance(parsed, dict):
        return []

    rels = parsed.get("relationships", [])
    if not isinstance(rels, list):
        return []

    out = []
    for rel in rels:
        if not isinstance(rel, dict):
            continue
        norm = _normalize_rel(
            rel.get("subject", ""),
            rel.get("predicate", ""),
            rel.get("object", ""),
            rel.get("confidence", 0.7),
        )
        if norm:
            out.append(norm)

    return out[:5]

def _make_prompt(item):
    summary_text = str(item["summary"])[:760]
    entities_text = json.dumps(item.get("entities", {}), ensure_ascii=False)[:420]
    return REL_PROMPT_TEMPLATE.format(
        summary=summary_text,
        entities=entities_text,
    )

def _generate_batch(prompts, max_new_tokens=256):
    return summarizer(
        prompts,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        use_cache=False,
    )

records_ds = Dataset.from_list(summary_entities)
REL_BATCH_SIZE = 2
kg_records = []

for i in tqdm(range(0, len(records_ds), REL_BATCH_SIZE), desc="Extracting relationships (batched)"):
    batch = records_ds.select(range(i, min(i + REL_BATCH_SIZE, len(records_ds))))
    items = [batch[j] for j in range(len(batch))]
    prompts = [_make_prompt(item) for item in items]

    outputs = _generate_batch(prompts, max_new_tokens=256)

    for item, output in zip(items, outputs):
        raw_text = _generated_text_from_output(output)
        relationships = _parse_relationships(raw_text)
        kg_records.append({
            "summary": item["summary"],
            "entities": item.get("entities", {}),
            "relationships": relationships,
            "source": item["source"],
        })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Sample 1:")
print(json.dumps(kg_records[0], indent=2, ensure_ascii=False))
print("\nSample 2:")
print(json.dumps(kg_records[min(1, len(kg_records)-1)], indent=2, ensure_ascii=False))

Extracting relationships (batched):  10%|█         | 1/10 [01:16<11:26, 76.24s/it]

In [33]:
# Debug one raw relationship-generation output (OOM-safe)
import gc
import torch

example_item = summary_entities[0]
mini_item = {
    "summary": str(example_item["summary"])[:260],
    "entities": {},
}
example_prompt = _make_prompt(mini_item)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

raw_example_output = summarizer(
    example_prompt,
    max_new_tokens=40,
    do_sample=False,
    return_full_text=False,
    use_cache=False,
)

raw_example_text = _generated_text_from_output(raw_example_output)
print("=== RAW MODEL OUTPUT ===")
print(raw_example_text)
print("\n=== PARSED RELATIONSHIPS ===")
print(json.dumps(_parse_relationships(raw_example_text), indent=2, ensure_ascii=False))

=== RAW MODEL OUTPUT ===
Relationships extracted from the summary:

```json
{"relationships":[{"subject":"kim_zolciak","predicate":"sued","object":"jillian_green"},{"subject":"kim_zolciak","

=== PARSED RELATIONSHIPS ===
[]
